# 2.5 自动微分 (Automatic Differentiation)

本 Notebook 包含《动手学深度学习》(D2L) 第 2.5 节的全部实战代码与课后练习题验证。
涵盖：**标量反向传播、梯度清零、非标量求导、detach分离计算、动态控制流求导** 以及 **练习题5（基于纯自动求导绘制正弦导函数）**。

## 2.5.1 一个简单的例子 (标量反向传播与梯度清零)
对目标函数 $y = 2 \mathbf{x}^\top \mathbf{x}$ 关于向量 $\mathbf{x}$ 求导，理论梯度应为 $4\mathbf{x}$。

In [1]:
# ==============================================================================
# 【代码 2.5.1-1】创建变量并开启梯度追踪 (requires_grad)
# ==============================================================================
import torch

x = torch.arange(4.0, requires_grad=True)
print("初始 x:", x)
print("默认梯度:", x.grad)  # 默认值为 None
# ==============================================================================

In [2]:
# ==============================================================================
# 【代码 2.5.1-2】前向传播计算 y 并反向传播 (backward)
# ==============================================================================
y = 2 * torch.dot(x, x)
print("y =", y)

y.backward()
print("求导后的 x.grad:", x.grad)
print("验证是否等于 4*x:", x.grad == 4 * x)
assert torch.equal(x.grad, 4 * x)
# ==============================================================================

In [3]:
# ==============================================================================
# 【代码 2.5.1-3】清除累计梯度 (zero_())
# 核心规则：PyTorch 默认会累加梯度，更新前必须手动清零！
# ==============================================================================
x.grad.zero_()  # 梯度就地清零

y = x.sum()
y.backward()
print("y=x.sum() 的导数:", x.grad)
assert torch.equal(x.grad, torch.ones_like(x))
# ==============================================================================

## 2.5.2 非标量变量的反向传播
当 $y$ 是向量而非标量时，直接调用 `y.backward()` 会报错。在深度学习中，必须先将其转化为标量损失（通常求和）后再反向传播。

In [4]:
# ==============================================================================
# 【代码 2.5.2-1】非标量反向传播 (先求和转化为标量)
# ==============================================================================
x.grad.zero_()
y = x * x  # y 是向量: [0, 1, 4, 9]

# 对非标量求和转为标量损失，再执行 backward
y.sum().backward()
print("y = x*x 的导数 (2*x):", x.grad)
assert torch.equal(x.grad, 2 * x)
# ==============================================================================

## 2.5.3 分离计算 (detach)
使用 `.detach()` 将中间变量从计算图中剥离，使其作为常数处理，梯度不再向后回传。

In [5]:
# ==============================================================================
# 【代码 2.5.3-1】detach 分离计算图
# ==============================================================================
x.grad.zero_()
y = x * x
u = y.detach()  # u 拥有 y 的数值，但不记录计算图历史

z = u * x
z.sum().backward()

# 因为 u 被当作常数，对 z = u*x 求关于 x 的导数等于 u
print("x.grad == u:", x.grad == u)
assert torch.equal(x.grad, u)
# ==============================================================================

## 2.5.4 Python 控制流的梯度计算
PyTorch 的动态计算图天然支持原生的 `while` 循环和 `if-else` 分支。

In [6]:
# ==============================================================================
# 【代码 2.5.4-1】动态循环与分支函数的梯度计算
# ==============================================================================
def f(a):
    b = a * 2
    while b.norm() < 1000:
        b = b * 2
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c

a = torch.randn(size=(), requires_grad=True)
d = f(a)
d.backward()

print("a.grad == d / a:", a.grad == d / a)
assert torch.isclose(a.grad, d / a)
# ==============================================================================

---
## 2.5 课后练习题 5 实验：基于纯自动微分绘制正弦函数导数
设 $f(x) = \sin(x)$，不使用解析导数公式 $\cos(x)$，而是完全利用 PyTorch 的 `x.grad` 绘制出导函数图像并验证。

In [7]:
# ==============================================================================
# 【练习 5】自动求导绘制正弦函数导数曲线
# ==============================================================================
import matplotlib.pyplot as plt

x = torch.linspace(-torch.pi, torch.pi, 200, requires_grad=True)
y = torch.sin(x)
y.sum().backward()

plt.figure(figsize=(6, 3.5))
plt.plot(x.detach().numpy(), y.detach().numpy(), label='f(x) = sin(x)')
plt.plot(x.detach().numpy(), x.grad.numpy(), '--', label="df/dx (Autograd)", color='red')
plt.axhline(0, color='gray', linewidth=0.5, linestyle=':')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.title("Autograd Derivative of sin(x)")
plt.show()
# ==============================================================================